### Ingest the data

### use wines-cluster

In [0]:
try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    pass  # already available in Databricks UI

username = "doman.mat@gmail.com"                     
schema = "wines/tables"                                                  
file_name = "wine_reviews-2026-04-11-23-44.csv"

file_path = f"/Workspace/Users/{username}/{schema}/{file_name}"

df = spark.read.csv(file_path, header=True, inferSchema=True)

Databricks data profile. Run in Databricks to view.

In [0]:
print(f"Row count: {df.count():,}")
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
display(df)

Databricks data profile. Run in Databricks to view.

# Save as Delta Bronze

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("wine_reviews_bronze")

print("Bronze table written.")

## Cleaning and casting 

In [0]:
from pyspark.sql.functions import col, expr, when                      

nulls_before = df.filter(col("alcohol").isNull()).count()              
 
df_cast = (df                                                          
    .withColumn("alcohol_cast", expr("try_cast(alcohol as double)"))
    .withColumn("alcohol_error", when(
        col("alcohol_cast").isNull() & col("alcohol").isNotNull(),
col("alcohol")
    ))
    .withColumn("alcohol", col("alcohol_cast"))
    .drop("alcohol_cast")
)

nulls_after = df_cast.filter(col("alcohol").isNull()).count()
cast_failures = nulls_after - nulls_before

print(f"Nulls before cast:  {nulls_before:,}")
print(f"Nulls after cast:   {nulls_after:,}")
print(f"Cast failures:      {cast_failures:,}")

if cast_failures > 0:
    df_cast.filter(col("alcohol_error").isNotNull()).select("alcohol_error").distinct().show()

In [0]:
from pyspark.sql.functions import col, expr, when                      

cast_columns = [                                                       
    ("alcohol",         "double"),
    ("vintage",         "int"),                                        
    ("case_production", "long"),
    ("retail",          "double"),
    ("rating",          "int"),
]

nulls_before = {c: df.filter(col(c).isNull()).count() for c, _ in
cast_columns}

df_cast = df
for column, dtype in cast_columns:
    df_cast = (df_cast
        .withColumn(f"{column}_cast", expr(f"try_cast({column} as {dtype})"))
        .withColumn(f"{column}_error", when(
            col(f"{column}_cast").isNull() & col(column).isNotNull(),col(column)
        ))
        .withColumn(column, col(f"{column}_cast"))
        .drop(f"{column}_cast")
    )

for column, _ in cast_columns:
    nulls_after = df_cast.filter(col(column).isNull()).count()
    failures = nulls_after - nulls_before[column]
    print(f"{column:20s}  nulls before={nulls_before[column]:>6,} after={nulls_after:>6,}  failures={failures:>6,}")
    if failures > 0:
        df_cast.filter(col(f"{column}_error").isNotNull()).select(f"{column}_error").distinct().show()